In [ ]:
# Get the working directory
import os
os.getcwd()

In [ ]:
# Read in the data

import pandas as pd
import numpy as np

# Paths to both Snapshot Terminology folders
intl_path = "../data/snomed/SnomedCT_InternationalRF2_PRODUCTION_20260201T120000Z/Snapshot/Terminology/"
ukcl_path = "../data/snomed/SnomedCT_UKClinicalRF2_PRODUCTION_20260506T000001Z/Snapshot/Terminology/"

# Load descriptions from both editions
print("Loading International descriptions...")
desc_international = pd.read_csv(
    intl_path + "sct2_Description_Snapshot-en_INT_20260201.txt",
    sep='\t',
    dtype={'conceptId': str, 'id': str},
    low_memory=False
)
print(f"International: {desc_international.shape}")

print("Loading UK Clinical descriptions...")
desc_ukclinical = pd.read_csv(
    ukcl_path + "sct2_Description_UKCLSnapshot-en_GB1000000_20260506.txt",
    sep='\t',
    dtype={'conceptId': str, 'id': str},
    low_memory=False
)
print(f"UK Clinical: {desc_ukclinical.shape}")

# Combine both
descriptions_combined = pd.concat(
    [desc_international, desc_ukclinical], 
    ignore_index=True
)
print(f"Combined: {descriptions_combined.shape}")

In [ ]:
# Filtering to the active descriptions and synonyms

# Step 1: Filter to active descriptions only
active_descriptions = descriptions_combined[
    descriptions_combined['active'] == 1
]
print(f"After active filter: {active_descriptions.shape}")

# Step 2: Filter to synonyms only (typeId 900000000000013009)
synonyms = active_descriptions[
    active_descriptions['typeId'] == 900000000000013009
]
print(f"After synonym filter: {synonyms.shape}")

# Step 3: Deduplicate — keep one description per conceptId
# keeping the first synonym encountered for each concept
clean_snomed = synonyms.drop_duplicates(subset='conceptId', keep='first')
print(f"After deduplication: {clean_snomed.shape}")

# Step 4: Keep only the columns we need
clean_snomed = clean_snomed[['conceptId', 'term']].reset_index(drop=True)
print(f"\nFinal dataset: {clean_snomed.shape}")
print(clean_snomed.head(10))

In [ ]:
# Check diabetes concepts are present
diabetes_check = clean_snomed[
    clean_snomed['term'].str.contains('type 2 diabetes', case=False, na=False)
]
print(f"Type 2 diabetes concepts found: {len(diabetes_check)}")
print(diabetes_check.head(20).to_string())

In [ ]:
# Save the dataset to disk

clean_snomed.to_csv(
    '../data/snomed/snomed_clean_descriptions.csv', 
    index=False
)
print("Saved successfully")

In [ ]:
# Initialse embedding

import chromadb
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the cleaned SNOMED dataset if not already in memory
clean_snomed = pd.read_csv('../data/snomed/snomed_clean_descriptions.csv',
                           dtype={'conceptId': str})
print(f"Loaded {len(clean_snomed)} concepts")

# Drop NaN rows
clean_snomed = clean_snomed.dropna(subset=['term', 'conceptId'])
print(f"Loaded {len(clean_snomed)} concepts")

# Load the embedding model
# all-MiniLM-L6-v2 is lightweight, fast, and well-suited for clinical text
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded")

# Initialise ChromaDB with persistent storage
# This saves vectors to disk so you never have to re-embed
chroma_client = chromadb.PersistentClient(
    path="../data/snomed/chroma_db"
)

# Create a collection — think of this as a table in a database
# If it already exists, delete and recreate for a clean run
try:
    chroma_client.delete_collection("snomed_concepts")
    print("Deleted existing collection")
except:
    pass

collection = chroma_client.create_collection(
    name="snomed_concepts",
    metadata={"hnsw:space": "cosine"}
)
print("ChromaDB collection created")

In [ ]:
# Embedding loop

# Embed and load in batches of 1000
# Batching prevents memory overflow on large datasets
BATCH_SIZE = 1000
total = len(clean_snomed)

print(f"Starting embedding of {total:,} concepts...")
print("This will take approximately 45-90 minutes on M1 Pro")
print("Do not close VS Code or let your Mac sleep\n")

for i in tqdm(range(0, total, BATCH_SIZE)):
    # Get current batch
    batch = clean_snomed.iloc[i:i+BATCH_SIZE]
    
    # Extract terms and IDs
    terms = batch['term'].tolist()
    ids = batch['conceptId'].tolist()
    
    # Generate embeddings for this batch
    embeddings = model.encode(terms, show_progress_bar=False)
    
    # Add to ChromaDB
    collection.add(
        embeddings=embeddings.tolist(),
        documents=terms,
        ids=ids
    )

print(f"\nEmbedding complete!")
print(f"Total concepts in ChromaDB: {collection.count()}")

In [ ]:
# Verification loop

# Verify the vector store works with a test query
test_query = "Type 2 diabetes mellitus"
query_embedding = model.encode([test_query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

print(f"Test query: '{test_query}'")
print("\nTop 5 retrieved concepts:")
for i, (doc, id_) in enumerate(zip(
    results['documents'][0], 
    results['ids'][0]
)):
    print(f"{i+1}. [{id_}] {doc}")

In [ ]:
help(chromadb)


In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path="../data/snomed/chroma_db"
)
print("ChromaDB initialised successfully")
print(f"Collections: {chroma_client.list_collections()}")